In [ ]:
import re
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.probability import FreqDist

text = """Books are a source of knowledge, imagination, and inspiration. They take readers on journeys across time, space, and emotions. From fiction to nonfiction, books help us understand the world and ourselves better. Reading regularly improves vocabulary, focus, and critical thinking. Whether for learning or leisure, books are lifelong companions."""

# 1. Lowercase + remove punctuation
cleaned_text = re.sub(r'[^\w\s]', '', text.lower())

# 2. Tokenization
sent_tokens = sent_tokenize(text)
word_tokens = word_tokenize(cleaned_text)

# 3. Python split vs word_tokenize
python_split = cleaned_text.split()
nltk_split = word_tokenize(cleaned_text)

# 4. Remove stopwords
stop_words = set(stopwords.words('english'))
filtered_tokens = [word for word in word_tokens if word not in stop_words]

# 5. Word frequency (excluding stopwords)
freq_dist = FreqDist(filtered_tokens)

print("Word Frequency:", freq_dist.most_common())

In [3]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet')

# 1. Words with alphabets only
alpha_words = re.findall(r'\b[a-zA-Z]+\b', cleaned_text)

# 2. Remove stopwords
alpha_filtered = [w for w in alpha_words if w not in stop_words]

# 3. Stemming
porter = PorterStemmer()
stemmed = [porter.stem(w) for w in alpha_filtered]

# 4. Lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(w) for w in alpha_filtered]

# 5. Compare
print("Stemmed:", stemmed)
print("Lemmatized:", lemmatized)


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\96597\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Stemmed: ['book', 'sourc', 'knowledg', 'imagin', 'inspir', 'take', 'reader', 'journey', 'across', 'time', 'space', 'emot', 'fiction', 'nonfict', 'book', 'help', 'us', 'understand', 'world', 'better', 'read', 'regularli', 'improv', 'vocabulari', 'focu', 'critic', 'think', 'whether', 'learn', 'leisur', 'book', 'lifelong', 'companion']
Lemmatized: ['book', 'source', 'knowledge', 'imagination', 'inspiration', 'take', 'reader', 'journey', 'across', 'time', 'space', 'emotion', 'fiction', 'nonfiction', 'book', 'help', 'u', 'understand', 'world', 'better', 'reading', 'regularly', 'improves', 'vocabulary', 'focus', 'critical', 'thinking', 'whether', 'learning', 'leisure', 'book', 'lifelong', 'companion']


In [6]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = [
    "Heavy rains disrupt daily life in Mumbai, local trains delayed.",
    "This wireless earbud set has amazing sound quality and battery life, but the case feels a bit flimsy.",
    "Just tried the new mogu-mogu — 10/10, super refreshing"
]

# 1. CountVectorizer
cv = CountVectorizer()
bow = cv.fit_transform(texts).toarray()

# TF-IDF Vectorization
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(texts)

feature_names = tfidf.get_feature_names()  

# Top 3 keywords per text
for i, row in enumerate(tfidf_matrix.toarray()):
    top3 = sorted(zip(feature_names, row), key=lambda x: -x[1])[:3]
    print(f"\nText {i+1} Top Keywords:", top3)



Text 1 Top Keywords: [('daily', 0.32311232743202956), ('delayed', 0.32311232743202956), ('disrupt', 0.32311232743202956)]

Text 2 Top Keywords: [('amazing', 0.2487839392343682), ('and', 0.2487839392343682), ('battery', 0.2487839392343682)]

Text 3 Top Keywords: [('10', 0.5427573398831146), ('mogu', 0.5427573398831146), ('just', 0.2713786699415573)]


In [19]:
text1 = "Artificial Intelligence focuses on learning patterns, making predictions, and adapting to new data."
text2 = "Blockchain ensures decentralized transactions, transparency, and security using cryptographic methods."

# a. Preprocess
t1_tokens = set(word_tokenize(re.sub(r'[^\w\s]', '', text1.lower())))
t2_tokens = set(word_tokenize(re.sub(r'[^\w\s]', '', text2.lower())))

# b. Jaccard Similarity
intersection = t1_tokens.intersection(t2_tokens)
union = t1_tokens.union(t2_tokens)
jaccard = len(intersection) / len(union)

# c. Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity
vec = TfidfVectorizer()
tfidf_sim = vec.fit_transform([text1, text2])
cos_sim = cosine_similarity(tfidf_sim[0:1], tfidf_sim[1:2])

print("Jaccard Similarity:", jaccard)
print("Cosine Similarity:", cos_sim[0][0])


Jaccard Similarity: 0.045454545454545456
Cosine Similarity: 0.04642927873353421


In [21]:
!pip install Pillow==9.5.0

In [30]:
from textblob import TextBlob
from wordcloud import WordCloud
import matplotlib.pyplot as plt

review = "The service was great, staff was friendly, and the aesthetics were amazing."

# 1. Sentiment
blob = TextBlob(review)
print("Polarity:", blob.polarity, "| Subjectivity:", blob.subjectivity)

# 2. Classification
if blob.polarity > 0:
    sentiment = "Positive"
elif blob.polarity < 0:
    sentiment = "Negative"
else:
    sentiment = "Neutral"
print("Sentiment:", sentiment)



Polarity: 0.5916666666666667 | Subjectivity: 0.7166666666666667
Sentiment: Positive


In [4]:
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
import numpy as np

text= """Books open doors to worlds unknown. They ignite imagination and foster empathy. Reading is a journey through time, space, and self. It’s both a mirror and a window."""

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
seqs = tokenizer.texts_to_sequences([text])[0]
inp = []
for i in range(1, len(seqs)):
    inp.append(seqs[:i+1])

inp = np.array(pad_sequences(inp, padding='pre'))
X, y = inp[:, :-1], inp[:, -1]

model = Sequential()
model.add(Embedding(len(tokenizer.word_index)+1, 10, input_length=X.shape[1]))
model.add(LSTM(50))
model.add(Dense(len(tokenizer.word_index)+1, activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
model.fit(X, y, epochs=100, verbose=0)

seed = "Technology"
seq = tokenizer.texts_to_sequences([seed])[0]
for _ in range(3):
    pad_seq = pad_sequences([seq], maxlen=X.shape[1], padding='pre')
    pred = model.predict(pad_seq, verbose=0).argmax()
    seq.append(pred)
inv_map = {v: k for k, v in tokenizer.word_index.items()}
print(" ".join([inv_map.get(i, "") for i in seq]))


AttributeError: module 'numpy' has no attribute 'typeDict'